# SigLIP 2 Vision-Language Pipeline — DIMER `MULTI-CAPABILITY` tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/siglip2-vision-language-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/siglip2-vision-language-pipeline/blob/main/tutorials/siglip2_vision_language_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fsiglip2--base--patch16--224-ffcc4d?style=flat)](https://huggingface.co/google/siglip2-base-patch16-224) [![Upstream](https://img.shields.io/badge/Upstream-huggingface%2Ftransformers-181717?style=flat&logo=github&logoColor=white)](https://github.com/huggingface/transformers) [![arXiv](https://img.shields.io/badge/arXiv-2502.14786-b31b1b.svg)](https://arxiv.org/abs/2502.14786)

**Profile:** `MULTI-CAPABILITY`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot image classification, image/text embeddings, cosine similarity, and text-to-image retrieval with the pinned `google/siglip2-base-patch16-224` checkpoint

**This notebook is standalone.** It carries the repository's package (4 modules under `src/siglip2_pipeline/`, at revision `8cee0461ef18`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `5ffaac51d5e2f3367f7dab0cad4be4cb07c0caa2` (~1539 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs every demonstrated capability locally in this kernel with its own input/output contract, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

This notebook uses the pinned `google/siglip2-base-patch16-224` checkpoint through the repository's public API — carried in this notebook — for zero-shot classification, image/text embeddings, cosine similarity, and text-to-image retrieval. **No gradient training, fine-tuning, in-context conditioning, or fitted preprocessing state occurs** — **no adaptation occurs.** Upstream provides the pretrained model/processor; this repository adds immutable pinning, integrity verification, safe local loading, stable inference contracts, the `validate_inputs` / `evaluation_report` role stages, exports, and provenance. Outputs are uncalibrated inference/ranking evidence. The default data are deterministic **synthetic tutorial/smoke assets** generated in code, not benchmark data.

**Learning objectives:** install the pinned runtime, read what the carried package guarantees, resolve and digest-verify the immutable model revision, generate the three synthetic sample images and validate them into an input manifest, run all five public operations (classification, image embeddings, text embeddings, similarity, retrieval) and read each capability's input/output contract, produce an evaluation report whose `top1_accuracy` and `recall_at_1` are `sample-sanity` evidence on the synthetic set, exercise an optional BYOD path and a default new-data path, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** object detection, semantic segmentation, OCR, caption generation, calibrated probabilities, universal thresholds, or production serving. GPU execution is outside this release contract: the tutorial runs on CPU even on a GPU host.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The release reference is **CPU-only**: the pinned `torch==2.14.0` install is the largest download of the run and the model is loaded with `device="cpu"` even on a GPU host; GPU execution requires a separately pinned/tested environment. The verified weight file is 1,500,800,904 bytes.
- **Knowledge:** basic Python and PIL image handling; what a sigmoid score and a cosine similarity are.
- **Data:** the default sample is three deterministic 32×32 synthetic images (red square, green circle, blue triangle) generated in code and digest-asserted, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (any colour mode; the pipeline converts to RGB and applies the pinned 224×224 image contract), plus your own candidate labels; model-bound text is lowercased and truncated to a 64-token maximum; candidate labels/queries must be non-empty; HTTP(S) image URLs are rejected. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send image contents to a hosted inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/siglip2-base-patch16-224` snapshot (~1539 MB in total) at revision `5ffaac51d5e2…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
]
NOTEBOOK_SOURCE = {
    'repository': 'siglip2-vision-language-pipeline',
    'repository_revision': '8cee0461ef18543b05e15434c5aaecfdf02c2eae',
    'embedded_module': 'src/siglip2_pipeline/config.py',
    'embedded_modules': ['src/siglip2_pipeline/config.py', 'src/siglip2_pipeline/model.py', 'src/siglip2_pipeline/provenance.py', 'src/siglip2_pipeline/pipeline.py'],
    'module_sha256': 'd7ba2f254f752e8419c13306db6b361721b5a2afa196cab0aef9f23d46aa56df',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/siglip2_pipeline/` @ `8cee0461ef18`)

The next 4 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (2 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/4:** `src/siglip2_pipeline/config.py`

In [ ]:
from __future__ import annotations

MODEL_ID = "google/siglip2-base-patch16-224"
MODEL_REVISION = "5ffaac51d5e2f3367f7dab0cad4be4cb07c0caa2"
MODEL_FILENAME = "model.safetensors"
MODEL_SHA256 = "612923381c76ec5a9bed335d1c48827e3f2e506ac31b044b63b2031fadee6a0b"
MODEL_SIZE_BYTES = 1_500_800_904
MODEL_LICENSE = "Apache-2.0"

DEFAULT_MODEL_KEY = "siglip2-base-patch16-224"
UNSAFE_WEIGHT_EXTENSIONS = (
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".h5",
    ".msgpack",
)

ALLOWED_CHECKPOINT_FILES = (
    "config.json",
    MODEL_FILENAME,
    "preprocessor_config.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer.model",
    "tokenizer_config.json",
)

DEFAULT_PROMPT_TEMPLATE = "This is a photo of {label}."
TEXT_MAX_LENGTH = 64

**Module 2/4:** `src/siglip2_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from collections.abc import Callable
from pathlib import Path
from typing import Any

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoProcessor

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

MANIFEST_NAME = "dimer-base-manifest.json"
#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in a repository-
#: local snapshot directory named by the model key and described by the committed manifest; a
#: standalone notebook carries that manifest inline and stages/verifies a working-directory copy.
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / DEFAULT_MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_checkpoint(
    snapshot_path: str | Path,
    *,
    require_configs: bool = False,
    return_manifest_verified: bool = False,
) -> Path | tuple[Path, bool]:
    root = Path(snapshot_path)
    if not root.is_dir():
        raise RuntimeError(f"Checkpoint directory does not exist: {root}")

    weight_path = root / MODEL_FILENAME
    if not weight_path.is_file():
        raise RuntimeError(f"Pinned checkpoint is missing {MODEL_FILENAME}")

    unsafe = sorted(
        p.name
        for p in root.iterdir()
        if p.is_file() and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS
    )
    if unsafe:
        raise RuntimeError(f"Refusing unsafe weight files: {unsafe}")

    manifest_path = root / MANIFEST_NAME
    manifest_verified = False

    if manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc

        files = manifest.get("files") or []
        if not files:
            raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")

        for entry in files:
            rel_path = entry.get("path")
            if not rel_path:
                continue
            target = root / rel_path
            if not target.is_file():
                raise RuntimeError(f"Manifest file missing: {rel_path}")
            exp_bytes = entry.get("bytes")
            if exp_bytes is not None and target.stat().st_size != exp_bytes:
                raise RuntimeError(
                    f"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}"
                )
            exp_sha = entry.get("sha256")
            if exp_sha is not None and _sha256(target) != exp_sha:
                raise RuntimeError(f"SHA-256 mismatch for {rel_path}")

        manifest_verified = True

    size = weight_path.stat().st_size
    if size != MODEL_SIZE_BYTES:
        raise RuntimeError(
            f"Unexpected {MODEL_FILENAME} size: {size}; expected {MODEL_SIZE_BYTES}"
        )

    digest = _sha256(weight_path)
    if digest != MODEL_SHA256:
        raise RuntimeError(
            f"Unexpected {MODEL_FILENAME} SHA-256: {digest}; expected {MODEL_SHA256}"
        )

    if require_configs:
        for req in ("config.json", "preprocessor_config.json"):
            if not (root / req).is_file():
                raise RuntimeError(f"Missing required configuration file: {req}")

    if return_manifest_verified:
        return root, manifest_verified
    return root


def _read_manifest(root: Path) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except ValueError as exc:
        raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing"
        )
    if not manifest.get("files"):
        raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.

    The identity in the manifest must be the pinned one; every manifest entry is then size- and
    SHA-256-checked by :func:`verify_checkpoint` (the existing verifier, which also asserts the
    weight file's pinned digest and byte count and refuses unsafe formats). Returns
    ``{"path": ..., **manifest}``.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    _, manifest_verified = verify_checkpoint(
        root, require_configs=True, return_manifest_verified=True
    )
    if not manifest_verified:
        raise RuntimeError(f"manifest at {root} was not verified")  # pragma: no cover
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and
    the small files but git-ignores the weights). Returns the relative paths fetched;
    :func:`verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def resolve_weights_path(
    weights_path: str | Path | None = None,
    cache_dir: str | Path | None = None,
) -> tuple[Path, str]:
    """Resolve weights path with precedence:

    1. Explicit argument `weights_path` -> 'explicit_path'
    2. Environment variable `SIGLIP2_WEIGHTS_DIR` -> 'env_var'
    3. Source checkout convention `weights/siglip2-base-patch16-224` -> 'repo_offline'
       (only if pyproject.toml exists at repo root and weights/ contains model.safetensors)
    4. Hugging Face Hub snapshot download -> 'hf_hub'
    """
    if weights_path is not None:
        return Path(weights_path), "explicit_path"

    env_dir = os.environ.get("SIGLIP2_WEIGHTS_DIR")
    if env_dir:
        return Path(env_dir), "env_var"

    repo_root = Path.cwd()  # standalone rewrite (build_notebook.py): no repository checkout exists; the notebook passes weights_dir explicitly
    if (repo_root / "pyproject.toml").is_file():
        repo_weights = repo_root / "weights" / DEFAULT_MODEL_KEY
        if (repo_weights / MODEL_FILENAME).is_file():
            return repo_weights, "repo_offline"

    hub_path = Path(
        snapshot_download(
            repo_id=MODEL_ID,
            revision=MODEL_REVISION,
            allow_patterns=list(ALLOWED_CHECKPOINT_FILES),
            cache_dir=str(cache_dir) if cache_dir is not None else None,
        )
    )
    return hub_path, "hf_hub"


_resolve_weights_path = resolve_weights_path


def _resolve_device(device: str | torch.device | None) -> torch.device:
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_components(
    *,
    device: str | torch.device | None = None,
    cache_dir: str | Path | None = None,
    weights_path: str | Path | None = None,
    return_metadata: bool = False,
) -> tuple[Any, Any, torch.device, Path] | tuple[Any, Any, torch.device, Path, dict[str, Any]]:
    """Acquire, verify, and load the one supported SigLIP 2 checkpoint."""

    candidate_path, source = resolve_weights_path(
        weights_path=weights_path,
        cache_dir=cache_dir,
    )

    verified, manifest_verified = verify_checkpoint(
        candidate_path,
        require_configs=True,
        return_manifest_verified=True,
    )
    target_device = _resolve_device(device)

    processor = AutoProcessor.from_pretrained(
        verified,
        local_files_only=True,
        trust_remote_code=False,
    )
    model = AutoModel.from_pretrained(
        verified,
        local_files_only=True,
        trust_remote_code=False,
        use_safetensors=True,
    )
    model = model.eval().to(target_device)

    weight_file = verified / MODEL_FILENAME
    metadata: dict[str, Any] = {
        "checkpoint_path": verified,
        "checkpoint_source": source,
        "manifest_verified": manifest_verified,
        "weight_sha256": _sha256(weight_file),
        "weight_size_bytes": weight_file.stat().st_size,
        "device": str(target_device),
    }

    if return_metadata:
        return model, processor, target_device, verified, metadata
    return model, processor, target_device, verified

**Module 3/4:** `src/siglip2_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

_RUNTIME_PACKAGES = (
    "huggingface-hub",
    "numpy",
    "pillow",
    "safetensors",
    "torch",
    "transformers",
)


def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def build_provenance(
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    prompt_template: str = DEFAULT_PROMPT_TEMPLATE,
    include_runtime: bool = True,
) -> dict[str, Any]:
    checkpoint_source = None
    manifest_verified = False
    weight_file = MODEL_FILENAME
    weight_sha256 = MODEL_SHA256
    weight_size = MODEL_SIZE_BYTES
    device_str = None
    resolved_checkpoint_path = None

    if pipeline is not None:
        if hasattr(pipeline, "checkpoint_path") and pipeline.checkpoint_path is not None:
            resolved_checkpoint_path = str(pipeline.checkpoint_path)
        if hasattr(pipeline, "checkpoint_source"):
            checkpoint_source = pipeline.checkpoint_source
        if hasattr(pipeline, "manifest_verified"):
            manifest_verified = bool(pipeline.manifest_verified)
        if hasattr(pipeline, "weight_sha256") and pipeline.weight_sha256:
            weight_sha256 = pipeline.weight_sha256
        if hasattr(pipeline, "weight_size_bytes") and pipeline.weight_size_bytes:
            weight_size = pipeline.weight_size_bytes
        if hasattr(pipeline, "device"):
            device_str = str(pipeline.device)

    if checkpoint_path is not None:
        resolved_checkpoint_path = str(checkpoint_path)
        if checkpoint_source is None:
            checkpoint_source = "explicit_path"

    model_record: dict[str, Any] = {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "weight_file": weight_file,
        "weight_sha256": weight_sha256,
        "weight_size_bytes": weight_size,
    }
    if checkpoint_source is not None:
        model_record["checkpoint_source"] = checkpoint_source
    if resolved_checkpoint_path is not None:
        model_record["checkpoint_path"] = resolved_checkpoint_path
    if pipeline is not None or checkpoint_path is not None:
        model_record["manifest_verified"] = manifest_verified

    inference_record: dict[str, Any] = {
        "prompt_template": prompt_template,
        "zero_shot_score_semantics": "independent_sigmoid_not_calibrated_probability",
        "embedding_normalization": "l2",
        "similarity": "cosine_via_normalized_dot_product",
    }
    if device_str is not None:
        inference_record["device"] = device_str

    provenance: dict[str, Any] = {
        "schema_version": 1,
        "model": model_record,
        "processor": {
            "input_resolution": [224, 224],
            "text_max_length": TEXT_MAX_LENGTH,
            "lowercase_model_bound_text": True,
        },
        "inference": inference_record,
    }
    if include_runtime:
        provenance["runtime"] = {
            "python": platform.python_version(),
            "implementation": platform.python_implementation(),
            "platform": sys.platform,
            "packages": {name: _package_version(name) for name in _RUNTIME_PACKAGES},
        }
    return provenance


def write_provenance(
    path: str | Path,
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    prompt_template: str = DEFAULT_PROMPT_TEMPLATE,
    include_runtime: bool = True,
) -> Path:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            build_provenance(
                pipeline=pipeline,
                checkpoint_path=checkpoint_path,
                prompt_template=prompt_template,
                include_runtime=include_runtime,
            ),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    return target

**Module 4/4:** `src/siglip2_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass
from io import BytesIO
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import DEFAULT_PROMPT_TEMPLATE, MODEL_ID, MODEL_REVISION, TEXT_MAX_LENGTH` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import load_components, stage_missing_files, verify_snapshot` removed — names are kernel globals defined by the carried modules

ImageInput = str | Path | bytes | Image.Image


@dataclass(frozen=True, slots=True)
class ClassificationScore:
    label: str
    score: float


@dataclass(frozen=True, slots=True)
class RetrievalHit:
    index: int
    score: float


def _coerce_image(value: ImageInput) -> Image.Image:
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, bytes):
        with Image.open(BytesIO(value)) as image:
            return image.convert("RGB")
    path = Path(value)
    raw = str(value)
    if raw.lower().startswith(("http://", "https://")):
        raise ValueError("Remote image URLs are not accepted; provide local bytes or a local path")
    with Image.open(path) as image:
        return image.convert("RGB")


def _require_texts(values: Sequence[str], *, name: str) -> list[str]:
    result = [value.strip() for value in values]
    if not result or any(not value for value in result):
        raise ValueError(f"{name} must contain at least one non-empty string")
    return result


def _siglip2_texts(values: Sequence[str]) -> list[str]:
    """Match SigLIP 2 training-time text lowercasing before tokenization."""

    return [value.lower() for value in values]


def _move_batch(batch: Any, device: torch.device) -> Any:
    if hasattr(batch, "to"):
        return batch.to(device)
    return {
        key: value.to(device) if hasattr(value, "to") else value
        for key, value in batch.items()
    }


class Siglip2Pipeline:
    def __init__(
        self,
        model: Any,
        processor: Any,
        *,
        device: str | torch.device = "cpu",
        checkpoint_path: Path | str | None = None,
        checkpoint_source: str | None = None,
        manifest_verified: bool = False,
        weight_sha256: str | None = None,
        weight_size_bytes: int | None = None,
    ) -> None:
        self.model = model
        self.processor = processor
        self.device = torch.device(device)
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        self.checkpoint_source = checkpoint_source
        self.manifest_verified = manifest_verified
        self.weight_sha256 = weight_sha256
        self.weight_size_bytes = weight_size_bytes

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | torch.device | None = None,
        cache_dir: str | Path | None = None,
        weights_path: str | Path | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Siglip2Pipeline:
        """Load the one supported checkpoint.

        ``weights_dir`` names a fleet snapshot directory holding ``dimer-base-manifest.json``
        (normally ``weights/<DEFAULT_MODEL_KEY>/``): manifest entries that are absent are staged
        with :func:`stage_missing_files` (only when ``allow_download=True``), the directory is
        verified against the manifest and the pinned digests by :func:`verify_snapshot`, and the
        same :func:`load_components` call then loads it as an explicit path (source
        ``explicit_path``; nothing goes through ``snapshot_download``).
        """
        if weights_dir is not None:
            if weights_path is not None:
                raise ValueError("pass either weights_dir or weights_path, not both")
            stage_missing_files(weights_dir, allow_download=allow_download)
            verify_snapshot(weights_dir)
            weights_path = weights_dir
        model, processor, target_device, _, metadata = load_components(
            device=device,
            cache_dir=cache_dir,
            weights_path=weights_path,
            return_metadata=True,
        )
        return cls(
            model,
            processor,
            device=target_device,
            checkpoint_path=metadata.get("checkpoint_path"),
            checkpoint_source=metadata.get("checkpoint_source"),
            manifest_verified=metadata.get("manifest_verified", False),
            weight_sha256=metadata.get("weight_sha256"),
            weight_size_bytes=metadata.get("weight_size_bytes"),
        )

    def zero_shot_classify(
        self,
        image: ImageInput,
        labels: Sequence[str],
        *,
        prompt_template: str = DEFAULT_PROMPT_TEMPLATE,
    ) -> list[ClassificationScore]:
        if not labels:
            raise ValueError("labels must contain at least one non-empty string")
        original_labels = list(labels)
        cleaned_labels = [label.strip() for label in original_labels]
        if any(not label for label in cleaned_labels):
            raise ValueError("labels must contain at least one non-empty string")
        if "{label}" not in prompt_template:
            raise ValueError("prompt_template must contain the literal {label} placeholder")

        prompts = [prompt_template.format(label=label) for label in cleaned_labels]
        batch = self.processor(
            text=_siglip2_texts(prompts),
            images=[_coerce_image(image)],
            padding="max_length",
            max_length=TEXT_MAX_LENGTH,
            return_tensors="pt",
        )
        batch = _move_batch(batch, self.device)
        with torch.inference_mode():
            logits = self.model(**batch).logits_per_image[0]
            scores = torch.sigmoid(logits).detach().cpu().tolist()

        ranked = [
            ClassificationScore(label=orig_label, score=float(score))
            for orig_label, score in zip(original_labels, scores, strict=True)
        ]
        return sorted(ranked, key=lambda item: item.score, reverse=True)

    def embed_image(self, images: Sequence[ImageInput]) -> np.ndarray:
        if not images:
            raise ValueError("images must contain at least one image")
        batch = self.processor(
            images=[_coerce_image(image) for image in images],
            return_tensors="pt",
        )
        batch = _move_batch(batch, self.device)
        with torch.inference_mode():
            features = self.model.get_image_features(**batch)
            features = F.normalize(features, p=2, dim=-1)
        return features.detach().cpu().numpy().astype(np.float32, copy=False)

    def embed_text(self, texts: Sequence[str]) -> np.ndarray:
        texts_list = _require_texts(texts, name="texts")
        batch = self.processor(
            text=_siglip2_texts(texts_list),
            padding="max_length",
            max_length=TEXT_MAX_LENGTH,
            return_tensors="pt",
        )
        batch = _move_batch(batch, self.device)
        with torch.inference_mode():
            features = self.model.get_text_features(**batch)
            features = F.normalize(features, p=2, dim=-1)
        return features.detach().cpu().numpy().astype(np.float32, copy=False)

    def similarity(self, images: Sequence[ImageInput], texts: Sequence[str]) -> np.ndarray:
        image_features = self.embed_image(images)
        text_features = self.embed_text(texts)
        return image_features @ text_features.T

    def retrieve(
        self,
        query: str,
        images: Sequence[ImageInput],
        *,
        top_k: int = 5,
    ) -> list[RetrievalHit]:
        if top_k < 1:
            raise ValueError("top_k must be >= 1")
        if not query.strip():
            raise ValueError("query must be a non-empty string")
        if not images:
            raise ValueError("images must contain at least one image")

        scores = self.similarity(images, [query])[:, 0]
        order = np.argsort(-scores, kind="stable")[: min(top_k, len(images))]
        return [RetrievalHit(index=int(index), score=float(scores[index])) for index in order]


def load_pipeline(
    *,
    device: str | torch.device | None = None,
    cache_dir: str | Path | None = None,
    weights_path: str | Path | None = None,
) -> Siglip2Pipeline:
    return Siglip2Pipeline.from_pretrained(
        device=device,
        cache_dir=cache_dir,
        weights_path=weights_path,
    )

# --------------------------------------------------------------------------
# Role stages (DIMER NOTEBOOK_SPEC 1.1 DAT24 / EVAL21)
#
# `validate_inputs` is the public validation stage: it applies exactly the checks the four core
# methods apply (image coercion through `_coerce_image`, non-empty texts through `_require_texts`,
# the label / prompt-template / top_k checks of `zero_shot_classify` and `retrieve`) and reports
# what was proven as an input manifest. `evaluation_report` is the public evaluation stage: its
# metric ids are this module's own helpers `top1_accuracy` and `recall_at_1` (the tutorial's
# sanity metrics, extracted from the notebook), and it always produces a report.
# --------------------------------------------------------------------------

#: The input contract and every named ceiling, in one readable structure.
INPUT_SCHEMA: dict[str, Any] = {
    "images": (
        "PIL.Image.Image, raw bytes, or a local path decodable by Pillow; any mode, converted to "
        "RGB; remote URLs are refused"
    ),
    "texts": "non-empty strings (labels, queries, or free text); lowercased before tokenization",
    "text_max_length": TEXT_MAX_LENGTH,
    "prompt_template": DEFAULT_PROMPT_TEMPLATE,
    "top_k": [1, None],
    "preprocessing": (
        "the pinned processor resizes every image to 224x224 and normalizes it; text is tokenized "
        f"to max_length={TEXT_MAX_LENGTH} with padding, longer text is truncated"
    ),
}


def _check_inputs(
    images: Sequence[ImageInput] | ImageInput | None,
    texts: Sequence[str] | None,
    *,
    top_k: int | None,
    prompt_template: str,
) -> tuple[list[Image.Image], list[str] | None]:
    """The checks the core methods apply, in their order, raising exactly what they raise."""
    if images is None:
        coerced: list[Image.Image] = []
    else:
        if isinstance(images, str | Path | bytes | Image.Image):
            images = [images]
        if not images:
            raise ValueError("images must contain at least one image")
        coerced = [_coerce_image(image) for image in images]
    checked_texts: list[str] | None = None
    if texts is not None:
        if not texts:
            raise ValueError("labels must contain at least one non-empty string")
        checked_texts = _require_texts(texts, name="texts")
    if "{label}" not in prompt_template:
        raise ValueError("prompt_template must contain the literal {label} placeholder")
    if top_k is not None and top_k < 1:
        raise ValueError("top_k must be >= 1")
    return coerced, checked_texts


def validate_inputs(
    images: Sequence[ImageInput] | ImageInput | None,
    texts: Sequence[str] | None = None,
    *,
    top_k: int | None = None,
    prompt_template: str = DEFAULT_PROMPT_TEMPLATE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``zero_shot_classify`` / ``embed_image`` /
    ``embed_text`` / ``retrieve`` would; a caller that wants the finding recorded catches the
    exception and stores ``str(exc)`` under ``findings``.
    """
    coerced, checked_texts = _check_inputs(
        images, texts, top_k=top_k, prompt_template=prompt_template
    )
    if names is not None and len(names) != len(coerced):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"image-{i}",
                "mode": image.mode,
                "size": list(image.size),
            }
            for i, image in enumerate(coerced)
        ],
        "texts": checked_texts,
        "top_k": top_k,
        "prompt_template": prompt_template,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def top1_accuracy(
    classifications: Sequence[Sequence[ClassificationScore]], expected_labels: Sequence[str]
) -> float:
    """Fraction of images whose top-scoring label equals the expected label (tutorial sanity)."""
    if len(classifications) != len(expected_labels):
        raise ValueError("classifications and expected_labels must have the same length")
    if not classifications:
        raise ValueError("classifications must not be empty")
    pairs = zip(classifications, expected_labels, strict=True)
    hits = sum(int(scores[0].label == expected) for scores, expected in pairs)
    return hits / len(classifications)


def recall_at_1(
    retrievals: Sequence[Sequence[RetrievalHit]], expected_indices: Sequence[int]
) -> float:
    """Fraction of queries whose top-ranked hit is the expected image index (tutorial sanity)."""
    if len(retrievals) != len(expected_indices):
        raise ValueError("retrievals and expected_indices must have the same length")
    if not retrievals:
        raise ValueError("retrievals must not be empty")
    pairs = zip(retrievals, expected_indices, strict=True)
    hits = sum(int(ranked[0].index == expected) for ranked, expected in pairs)
    return hits / len(retrievals)


def evaluation_report(
    result: dict[str, Any],
    targets: dict[str, Any] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``result`` gathers what the notebook demonstrated: ``classifications`` (one
    ``zero_shot_classify`` list per image), ``retrievals`` (one ``retrieve`` list per query), and
    optionally ``embedding_shapes`` / ``similarity_shape``. ``targets`` supplies the ground truth:
    ``labels`` (the expected label per classified image) and/or ``retrieval_indices`` (the expected
    image index per query). With targets the report carries ``top1_accuracy`` (against the
    fixed-class baseline 1/n_labels) and/or ``recall_at_1`` with the verdict ``sample-sanity``;
    without them the verdict is ``not-measurable``. Embeddings and similarity matrices are
    representations with no intrinsic metric and are always reported as not measurable.
    """
    classifications = list(result.get("classifications") or [])
    retrievals = list(result.get("retrievals") or [])
    base: dict[str, Any] = {
        "task": "zero-shot image classification, image/text embeddings, similarity and retrieval",
        "score_semantics": (
            "classification and similarity scores are independent SigLIP sigmoids / cosine "
            "similarities, not calibrated probabilities and not summing to one; the decision rule "
            "used for the sanity metrics is argmax; no threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_classified_images": len(classifications),
        "n_retrieval_queries": len(retrievals),
        "embeddings": (
            "representations, not predictions: no intrinsic accuracy metric; "
            f"shapes {result.get('embedding_shapes')}"
        ),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    labels = (targets or {}).get("labels")
    indices = (targets or {}).get("retrieval_indices")
    metrics: list[dict[str, Any]] = []
    baselines: list[dict[str, Any]] = []
    estimation = "single synthetic sample set; no dispersion estimate"
    if labels is not None and classifications:
        n_candidates = len(classifications[0])
        metrics.append(
            {
                "id": "top1_accuracy",
                "value": top1_accuracy(classifications, list(labels)),
                "n_candidate_labels": n_candidates,
                "estimation": estimation,
            }
        )
        baselines.append(
            {
                "id": "fixed_class_baseline",
                "metric": "top1_accuracy",
                "value": 1.0 / n_candidates if n_candidates else None,
                "note": "always predicting one fixed candidate label",
            }
        )
    if indices is not None and retrievals:
        metrics.append(
            {
                "id": "recall_at_1",
                "value": recall_at_1(retrievals, list(indices)),
                "n_gallery_images": len(result.get("gallery_ids") or []) or None,
                "estimation": estimation,
            }
        )
    if not metrics:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no expected labels or expected retrieval indices were supplied",
            "needs": (
                "labelled images from the deployment domain (one expected label per image among "
                "the candidate labels) scored with top1_accuracy against the fixed-class baseline, "
                "and/or query-image relevance pairs scored with recall_at_1"
            ),
        }
    return {
        **base,
        "metrics": metrics,
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(classifications)} labelled image(s) and {len(retrievals)} query(ies) from the "
            "tutorial sample; not a benchmark"
        ),
        "needs": (
            "a labelled evaluation set from the deployment domain, with prompt wording fixed in "
            "advance, for any generalisable accuracy or retrieval claim"
        ),
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `5ffaac51d5e2…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Siglip2Pipeline.from_pretrained(device="cpu", weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "siglip2-base-patch16-224",
  "modelId": "google/siglip2-base-patch16-224",
  "revision": "5ffaac51d5e2f3367f7dab0cad4be4cb07c0caa2",
  "files": [
    {
      "path": "README.md",
      "bytes": 17268,
      "sha256": "b7ab37d51f75f1b945689b3a880cab56d27543edee3f433c7f40a238bc989134"
    },
    {
      "path": "config.json",
      "bytes": 253,
      "sha256": "fe8b5fe6d5734360678fd71c11c21e1ea3364bd8598d34295d9206335973ffd7"
    },
    {
      "path": "model.safetensors",
      "bytes": 1500800904,
      "sha256": "612923381c76ec5a9bed335d1c48827e3f2e506ac31b044b63b2031fadee6a0b"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 394,
      "sha256": "9b36b57ebaf20f09bf4c22100ccc21877ea6bfe5aead0c00c59f8af8ccefacfc"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 636,
      "sha256": "baec30ea10906f16adb8c18af7a34023002c1746542612b8b41c9f09e1351351"
    },
    {
      "path": "tokenizer.json",
      "bytes": 34363039,
      "sha256": "cb9140fae3ac5122c972d37adf83e1248471a38147ad76f8215c8872c6fd8322"
    },
    {
      "path": "tokenizer.model",
      "bytes": 4241003,
      "sha256": "61a7b147390c64585d6c3543dd6fc636906c9af3865a5548f27f31aee1d4c8e2"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 47164,
      "sha256": "14afe629fe4959b9e0d51e1852b8d9f7ad074f90a1a7125a4fcdd17f06e78fc8"
    }
  ],
  "totalBytes": 1539470661
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Siglip2Pipeline.from_pretrained(device="cpu", weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: three 32×32 RGB images — a red square, a green circle and a blue triangle on a light background — rendered in code as ASCII PPM files exactly as the repository's `examples/sample-data/generate_samples.py` writes them, so each file's SHA-256 is asserted against the digest the repository checks in. They provide deterministic tutorial ground truth (the expected label of each image) for the sanity metrics later; they are **synthetic tutorial/smoke assets**, not benchmark data. BYOD is optional and disabled by default; when enabled, upload one image (Colab) or set `BYOD_PATH` (other Jupyter environments), and edit `BYOD_LABELS` for the intended domain. Look for the three file names, sizes and digests.

In [ ]:
import hashlib
import io
import os
from pathlib import Path

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}
BYOD_LABELS = ["flooded street", "normal road", "fallen electrical pole"]
SAMPLE_DIGESTS = {  # examples/sample-data/SHA256SUMS
    "red_square.ppm": "b38ff0c9131677ed6cf09832eff40a22841e1a4725d426fad3b1bf6a1dbdb096",
    "green_circle.ppm": "2e3e657686a0f6a6df3f3621d21a410d20d9a47b109f06f9405faa4f747a663f",
    "blue_triangle.ppm": "f0f4c38c7af92b3a6b1d55a25272156edd87b41e029e116cfa059003dae029a3",
}
WIDTH, HEIGHT, BACKGROUND = 32, 32, (245, 245, 245)


def shape_mask(shape: str, x: int, y: int) -> bool:
    if shape == "square":
        return 8 <= x < 24 and 8 <= y < 24
    if shape == "circle":
        return (x - 16) ** 2 + (y - 16) ** 2 <= 9**2
    if not 7 <= y < 26:
        return False
    half = (y - 7) // 2
    return 16 - half <= x <= 16 + half


def render_ppm(foreground: tuple, shape: str) -> str:
    # The repository's generate_samples.py rendering: ASCII P3, 24 values per line.
    lines = ["P3", f"{WIDTH} {HEIGHT}", "255"]
    for y in range(HEIGHT):
        row = []
        for x in range(WIDTH):
            pixel = foreground if shape_mask(shape, x, y) else BACKGROUND
            row.extend(str(value) for value in pixel)
        for offset in range(0, len(row), 24):
            lines.append(" ".join(row[offset : offset + 24]))
    return "\n".join(lines) + "\n"


os.makedirs("outputs/sample-data", exist_ok=True)
specs = {"red_square.ppm": ((220, 40, 40), "square"), "green_circle.ppm": ((40, 170, 75), "circle"), "blue_triangle.ppm": ((40, 90, 220), "triangle")}
images = []
for name, (foreground, shape) in specs.items():
    path = Path("outputs/sample-data") / name
    path.write_bytes(render_ppm(foreground, shape).encode("ascii"))
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != SAMPLE_DIGESTS[name]:
        raise ValueError(f"Synthetic sample digest mismatch for {name}: {digest} != {SAMPLE_DIGESTS[name]}")
    images.append(path)
expected_labels = ["red square", "green circle", "blue triangle"]
candidate_labels = [*expected_labels, "abstract geometric shape"]
sample_kind = "synthetic"
sample_sha256 = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in images}
for path in images:
    with Image.open(path) as im:
        print({"name": path.name, "mode": im.mode, "size": im.size, "sha256": sample_sha256[path.name]})

byod_image = None
if USE_BYOD:
    try:
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one image.")
        name, data = next(iter(uploaded.items()))
        target = Path("outputs/byod") / Path(name).name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    except ModuleNotFoundError:
        if not BYOD_PATH:
            raise RuntimeError("Outside Colab, set BYOD_PATH before enabling BYOD.") from None
        target = Path(BYOD_PATH).expanduser()
    if not target.is_file():
        raise FileNotFoundError(target)
    try:
        with Image.open(target) as im:
            im.verify()
    except Exception as exc:
        raise ValueError(f"BYOD file is not a decodable image: {target}") from exc
    if not BYOD_LABELS or any(not x.strip() for x in BYOD_LABELS):
        raise ValueError("BYOD_LABELS must contain non-empty labels.")
    byod_image = target
    sample_kind = "BYOD"
    print({"byod_image": str(byod_image), "byod_labels": BYOD_LABELS})
else:
    print("BYOD disabled; default path is non-interactive.")

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks the five public operations apply — every image coerced to RGB the way `zero_shot_classify` / `embed_image` do (local bytes or paths only; HTTP(S) URLs are rejected), non-empty labels/queries, a `prompt_template` carrying the `{label}` placeholder, `top_k >= 1` — and returns an **input manifest** naming the schema and ceilings. The effective runtime and model identity (Python, PyTorch, Transformers, device, model id, immutable revision, verified weight digest and byte count) are printed first (the 224×224 image contract, the 64-token text maximum, the default prompt template), each image's observed mode and size, the texts, and the verdict. The manifest is written to `outputs/siglip2_vision_language_input_manifest.json`. To show what rejection looks like, the cell also validates a remote URL and records the pipeline's own error message as a finding. The ceilings are printed before the model runs anything.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", pipe.device)
print("Model ID:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Checkpoint source:", pipe.checkpoint_source, "| manifest verified:", pipe.manifest_verified)
print("Weight SHA-256:", pipe.weight_sha256)
print("Weight bytes:", pipe.weight_size_bytes)
print({'ceilings': {'TEXT_MAX_LENGTH': TEXT_MAX_LENGTH, 'DEFAULT_PROMPT_TEMPLATE': DEFAULT_PROMPT_TEMPLATE, 'image_contract': '224x224 RGB after processor resize', 'device': str(pipe.device)}})
validation_images = [*images] + ([byod_image] if byod_image is not None else [])
validation_names = [p.name for p in validation_images]
input_manifest = validate_inputs(validation_images, candidate_labels, top_k=len(images), names=validation_names)
# Demonstrate rejection on an input the pipeline refuses; the finding is recorded, not swallowed.
try:
    validate_inputs("https://example.invalid/not-allowed.png", candidate_labels)
except ValueError as exc:
    input_manifest["findings"].append({"input": "remote-url-probe", "verdict": "rejected", "message": str(exc)})
with open('outputs/siglip2_vision_language_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Capability A — zero-shot classification

`zero_shot_classify(image, labels)` scores each candidate label's prompt (`DEFAULT_PROMPT_TEMPLATE`, model-bound text lowercased) against the image and returns the labels **ranked by descending score**. Zero-shot scores are independent SigLIP sigmoids: **not calibrated probabilities** and not required to sum to one; they are **prompt/label dependent** — rewording a label changes its score. The tutorial uses argmax only for its sanity check; any deployment threshold/abstention rule belongs to the **downstream application** and must be calibrated on representative labelled data. Look for one ranked list per sample image and the BYOD ranking when enabled.

In [ ]:
from dataclasses import asdict

classifications = []
classification_rows = []
for image_path, expected in zip(images, expected_labels, strict=True):
    scores = pipe.zero_shot_classify(image_path, candidate_labels)
    classifications.append(scores)
    classification_rows.append({"image": image_path.name, "expected_label": expected, "predicted_label": scores[0].label, "scores": [asdict(x) for x in scores]})
    print(image_path.name, "->", [(s.label, round(s.score, 4)) for s in scores])

byod_scores = None
if byod_image is not None:
    byod_scores = pipe.zero_shot_classify(byod_image, BYOD_LABELS)
    print("BYOD top label:", byod_scores[0].label, [(s.label, round(s.score, 4)) for s in byod_scores])

## 7. Capability B — image and text embeddings, similarity

`embed_image` and `embed_text` return **L2-normalized** vectors, one per input, in a shared space: embeddings are representations, not predictions, and have **no intrinsic accuracy metric**. `similarity(images, texts)` is their cosine similarity matrix (rows = images, columns = texts); a similarity is a raw score, not a calibrated probability. Look for the two embedding shapes and the 3×3 matrix whose diagonal should dominate on the synthetic set.

In [ ]:
image_embeddings = pipe.embed_image(images)
text_embeddings = pipe.embed_text(expected_labels)
similarity = pipe.similarity(images, expected_labels)
print("Image embeddings:", image_embeddings.shape, "| L2 norms:", np.round(np.linalg.norm(image_embeddings, axis=1), 4).tolist())
print("Text embeddings:", text_embeddings.shape)
print("Similarity rows:", [p.name for p in images])
print("Similarity columns:", expected_labels)
print(np.array2string(similarity, precision=4))
byod_embedding = pipe.embed_image([byod_image]) if byod_image is not None else None

## 8. Capability C — text-to-image retrieval

`retrieve(query, images, top_k)` ranks the gallery images by cosine similarity to one text query and returns `RetrievalHit(index, score)` entries **ordered by descending score** (ties broken by stable index order). Retrieval usefulness on real data requires a downstream labelled evaluation; here each expected label is used as the query over the three-image gallery so the ranking is falsifiable.

In [ ]:
retrievals = []
retrieval_rows = []
for query, expected_image in zip(expected_labels, images, strict=True):
    hits = pipe.retrieve(query, images, top_k=len(images))
    retrievals.append(hits)
    retrieval_rows.append({"query": query, "expected_image": expected_image.name, "hits": [{"rank": r, "index": h.index, "filename": images[h.index].name, "score": h.score} for r, h in enumerate(hits, 1)]})
    print(query, "->", [(images[h.index].name, round(h.score, 4)) for h in hits])

## 9. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. On the synthetic set it carries `top1_accuracy` (the repository's helper: the fraction of images whose top-scoring label equals the expected label, compared with the fixed-class baseline of always predicting one candidate, 1/4 here) and `recall_at_1` (the fraction of queries whose top hit is the expected image) with the verdict `sample-sanity` — tiny synthetic sanity metrics, not estimates of generalization. Embeddings and similarity have no intrinsic metric and are reported as representations. Without expected labels or expected retrieval indices (the BYOD case) the verdict is `not-measurable` and the report states what labelled data would make the task measurable. The report is written to `outputs/siglip2_vision_language_evaluation_report.json`.

In [ ]:
result = {"classifications": classifications, "retrievals": retrievals, "gallery_ids": [p.name for p in images], "embedding_shapes": {"image": list(image_embeddings.shape), "text": list(text_embeddings.shape)}, "similarity_shape": list(similarity.shape)}
targets = {"labels": expected_labels, "retrieval_indices": list(range(len(images)))}
report = evaluation_report(result, targets, sample_kind=sample_kind)
with open('outputs/siglip2_vision_language_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if byod_scores is not None:
    byod_report = evaluation_report({"classifications": [byod_scores]}, sample_kind="BYOD")
    print("BYOD report verdict:", byod_report["verdict"], "-", byod_report["reason"])
top1_sanity_accuracy = next(m["value"] for m in report["metrics"] if m["id"] == "top1_accuracy")
retrieval_recall_at_1 = next(m["value"] for m in report["metrics"] if m["id"] == "recall_at_1")
print("Top-1 sanity accuracy:", top1_sanity_accuracy, "| fixed-class baseline:", report["baselines"][0]["value"], "| recall@1:", retrieval_recall_at_1)

## 10. Default new-data inference

A deterministic `yellow_square.ppm`, distinct from the three-image evaluation set, exercises the new-input path without interaction. It remains synthetic evidence and does not establish deployment accuracy.

In [ ]:
os.makedirs("outputs/new-data", exist_ok=True)
new_image_path = Path("outputs/new-data") / "yellow_square.ppm"
im = Image.new("RGB", (32, 32), "white")
px = im.load()
for y in range(8, 24):
    for x in range(8, 24):
        px[x, y] = (255, 255, 0)
im.save(new_image_path)
new_labels = ["yellow square", "blue circle", "red triangle", "abstract geometric shape"]
new_data_scores = pipe.zero_shot_classify(new_image_path, new_labels)
print("New-data top label:", new_data_scores[0].label, [(s.label, round(s.score, 4)) for s in new_data_scores])

## 11. Export outputs and provenance

Every demonstrated capability gets a stable export: `outputs/classification.json`, `outputs/similarity.csv` (rows = images, columns = expected labels), `outputs/retrieval.json`, `outputs/image_embeddings.npz` and `outputs/text_embeddings.npz` (input identifiers stored beside the vectors), `outputs/new_data_classification.json`, `outputs/metrics.json`, `outputs/provenance.json` (the package's own provenance: effective model, immutable revision, verified checkpoint, runtime packages, CPU device, inference semantics), and `outputs/siglip2_vision_language_result.json` with the input manifest, the evaluation report, the sample identity and digests, the notebook's source (repository, revision, embedded module digests, generator), the model identifier, the immutable model revision and licence, and the runtime identity. No trained/adapted model artifact is produced because this is pretrained inference. No credentials are recorded.

In [ ]:
import csv

with open("outputs/classification.json", "w", encoding="utf-8") as handle:
    json.dump(classification_rows, handle, indent=2)
with open("outputs/retrieval.json", "w", encoding="utf-8") as handle:
    json.dump(retrieval_rows, handle, indent=2)
with open("outputs/new_data_classification.json", "w", encoding="utf-8") as handle:
    json.dump({"image": new_image_path.name, "scores": [asdict(x) for x in new_data_scores]}, handle, indent=2)
with open("outputs/similarity.csv", "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["image", *expected_labels])
    for path, row in zip(images, similarity, strict=True):
        writer.writerow([path.name, *map(float, row)])
np.savez_compressed("outputs/image_embeddings.npz", image_ids=np.asarray([p.name for p in images]), vectors=image_embeddings)
np.savez_compressed("outputs/text_embeddings.npz", text_ids=np.asarray(expected_labels), vectors=text_embeddings)
metrics = {"evidence_type": "synthetic_tutorial_sanity_only", "sample_sha256": sample_sha256, "classification": {"metric": "top1_accuracy", "value": top1_sanity_accuracy, "fixed_class_baseline": report["baselines"][0]["value"]}, "retrieval": {"metric": "recall_at_1", "value": retrieval_recall_at_1}}
with open("outputs/metrics.json", "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
if byod_image is not None:
    with open("outputs/byod_classification.json", "w", encoding="utf-8") as handle:
        json.dump({"image": byod_image.name, "scores": [asdict(x) for x in byod_scores]}, handle, indent=2)
    np.savez_compressed("outputs/byod_image_embedding.npz", image_ids=np.asarray([byod_image.name]), vectors=byod_embedding)
write_provenance("outputs/provenance.json", pipeline=pipe)

payload = {
    'classification': classification_rows,
    'retrieval': retrieval_rows,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'names': [p.name for p in images], 'sha256': sample_sha256, 'expected_labels': expected_labels, 'candidate_labels': candidate_labels},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'device': str(pipe.device), 'checkpoint_source': pipe.checkpoint_source, 'manifest_verified': pipe.manifest_verified},
}
with open('outputs/siglip2_vision_language_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
expected_outputs = ["classification.json", "image_embeddings.npz", "text_embeddings.npz", "similarity.csv", "retrieval.json", "metrics.json", "new_data_classification.json", "provenance.json", "siglip2_vision_language_input_manifest.json", "siglip2_vision_language_evaluation_report.json", "siglip2_vision_language_result.json"]
missing = [n for n in expected_outputs if not (Path("outputs") / n).is_file()]
if missing:
    raise RuntimeError(f"Missing expected tutorial outputs: {missing}")
print(sorted(os.listdir("outputs")))

## Interpretation and limits

A successful run proves that the pinned runtime installs, the pinned checkpoint passes integrity checks, all five public operations execute on the validated inputs, and machine-readable outputs/provenance are produced. Successful execution proves that the recorded repository revision's package, carried in this notebook, can do exactly that — without the repository being reachable — and no more.

It **does not** prove production fitness, domain accuracy, fairness, robustness, calibration, latency, GPU compatibility, or universal thresholds; the evaluation report says `sample-sanity` on the synthetic set for that reason and `not-measurable` on BYOD. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain. SigLIP scores are prompt-dependent and uncalibrated: unexpected scores should first trigger review of prompt/label wording. Integrity/download failures must be fixed rather than bypassing pin/hash checks; for memory pressure, reduce images per call; a GPU host is expected to remain on CPU because the release lock is CPU-only.

**Next experiments:** enable `USE_BYOD` with non-sensitive data and your own labels; compare prompt templates for the same labels and watch the scores move; evaluate representative labelled real images with `top1_accuracy` against the fixed-class baseline and labelled real-image retrieval with `recall_at_1`; run a downstream evaluation of the exported embeddings; for deployment, define the production prompt/label policy, inspect failure modes/subgroups, calibrate thresholds or abstention rules, and benchmark the intended serving hardware.

## References

- Repository README: https://github.com/kurtvalcorza/siglip2-vision-language-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/siglip2-vision-language-pipeline/blob/main/MODEL_CARD.md
- Sample dataset card: https://github.com/kurtvalcorza/siglip2-vision-language-pipeline/blob/main/examples/sample-data/DATASET_CARD.md
- Upstream model: https://huggingface.co/google/siglip2-base-patch16-224
- Upstream library: https://github.com/huggingface/transformers
- SigLIP 2: Multilingual Vision-Language Encoders: https://arxiv.org/abs/2502.14786